# Chapter 1: Image Processing and CNN Fundamentals

Welcome to the first notebook of the Image Processing and CNN module. In this notebook, we will lay the foundation for understanding how computers "see" images and how we can use Deep Learning to classify them.

**Objective:**
1.  Understand images as numerical data (NumPy arrays).
2.  Perform basic image manipulations (cropping, channel separation).
3.  Prepare image data for Machine Learning (One-Hot Encoding).
4.  Build a baseline Image Classifier using a Multi-Layer Perceptron (MLP).
5.  Understand the limitations of MLPs for images and the need for Convolutional Neural Networks (CNNs).

---

## 1. Images as Data

To a computer, an image is not a visual scene but a grid of numbers. Each number represents the intensity of a pixel.

### 1.1 Pixels and Arrays
A digital image is made up of **pixels** (picture elements). 

- **Grayscale Image**: A 2D matrix where value `0` is black and `255` is white.
- **Color Image (RGB)**: A 3D matrix with three channels: Red, Green, and Blue.

![Image Representation as Matrix](https://res.cloudinary.com/practicaldev/image/fetch/s--c78X1K6o--/c_limit%2Cf_auto%2Cfl_progressive%2Cq_auto%2Cw_880/https://dev-to-uploads.s3.amazonaws.com/uploads/articles/k09590823485abs84jjd.png)
*Figure: An image is essentially a matrix of pixel values (0-255).* 
[Image Source](https://dev.to/)



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data  # Using skimage to load sample images easily

# Load a sample image provided by scikit-image
# 'chelsea' is a classic cat image used in image processing tutorials
image = data.chelsea()

# Display the image
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis('off')
plt.title("Sample Image: Chelsea the Cat")
plt.show()

### 1.2 Examining the Shape
Let's check the dimensions of this image. It is stored as a NumPy array.

In [ ]:
print(f"Type of image data: {type(image)}")
print(f"Shape of the image: {image.shape}")

# The shape is typically (Height, Width, Channels)
height, width, channels = image.shape
print(f"Height: {height} pixels")
print(f"Width: {width} pixels")
print(f"Channels: {channels} (Red, Green, Blue)")

### 1.3 Accessing Pixel Values
We can inspect the R, G, B values of a specific pixel using array indexing `[row, column]`.

In [ ]:
# Inspect a pixel at row 100, column 100
pixel_value = image[100, 100]
print(f"Pixel values at [100, 100] (RGB): {pixel_value}")

# Interpretation:
# The three numbers correspond to the intensity of Red, Green, and Blue light respectively.

---

## 2. Image Manipulation

Since images are just NumPy arrays, we can use standard slicing and arithmetic operations to modify them.

### 2.1 Modifying Channels
What happens if we set the Green and Blue channels to zero? We should be left with only the Red component.

In [ ]:
# Make a copy to avoid modifying the original
red_image = image.copy()

# Set Green (index 1) and Blue (index 2) channels to 0
red_image[:, :, 1] = 0  # Zero out Green
red_image[:, :, 2] = 0  # Zero out Blue

plt.figure(figsize=(6, 6))
plt.imshow(red_image)
plt.title("Red Channel Only")
plt.axis('off')
plt.show()

### 2.2 Cropping an Image
Cropping is simply slicing the array. Let's zoom in on the cat's face.

In [ ]:
# Select a region of interest (ROI)
# Format: image[start_row:end_row, start_col:end_col]
face_crop = image[50:250, 80:280]

plt.figure(figsize=(6, 6))
plt.imshow(face_crop)
plt.title("Cropped Face")
plt.axis('off')
plt.show()

### Exercise 1
**Goal:** Create a "Negative" of the cat image.

**Hint:** In an 8-bit image (0-255), the negative is obtained by subtracting the pixel value from the maximum value (255). 

*e.g., White (255) becomes Black (0), and Black (0) becomes White (255).*

In [ ]:
# TODO: Create the negative of the 'image' variable
# negative_image = ...

# Solution:
negative_image = 255 - image

plt.figure(figsize=(6, 6))
plt.imshow(negative_image)
plt.title("Negative Image")
plt.axis('off')
plt.show()

---

## 3. Classification Fundamentals: One-Hot Encoding

Before building a model, we need to prepare our target labels. Neural networks process numbers, but our labels might be categories like "Cat", "Dog", "Car". We also don't want to use integers like 1, 2, 3 because that implies an order (3 > 1), which isn't true for categories.

**One-Hot Encoding** solves this by converting each category into a binary vector.

| Label | Integer | One-Hot Vector |
| :--- | :--- | :--- |
| Red | 0 | `[1, 0, 0]` |
| Green | 1 | `[0, 1, 0]` |
| Blue | 2 | `[0, 0, 1]` |

![One Hot Encoding](https://miro.medium.com/max/1400/1*ggtP4a5YaRx6l09KQaYOnw.png)
*Figure: Visualizing One-Hot Encoding process.* 
[Image Source](https://medium.com/)

In [ ]:
import numpy as np
from tensorflow.keras.utils import to_categorical

# Suppose we have 3 classes: 0 (T-shirt), 1 (Dress), 2 (Shoe)
labels = np.array([0, 1, 2, 0])

print(f"Original Labels: {labels}")

# Apply One-Hot Encoding
one_hot_labels = to_categorical(labels, num_classes=3)

print("One-Hot Encoded Labels:")
print(one_hot_labels)

---

## 4. Case Study: Fashion MNIST Classification

We will now build a real image classifier. 
**Dataset:** Fashion MNIST. It consists of 70,000 grayscale images in 10 categories. The images show individual articles of clothing at low resolution (28 by 28 pixels).

Class Labels:
*   0: T-shirt/top
*   1: Trouser
*   2: Pullover
*   3: Dress
*   4: Coat
*   5: Sandal
*   6: Shirt
*   7: Sneaker
*   8: Bag
*   9: Ankle boot

In [ ]:
from tensorflow.keras.datasets import fashion_mnist

# Load the data
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

print(f"Training data shape: {train_images.shape}")
print(f"Test data shape: {test_images.shape}")

# Normalize pixel values to be between 0 and 1
# Neural networks converge faster with normalized inputs
train_images = train_images / 255.0
test_images = test_images / 255.0

In [ ]:
# Visualize one sample
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i], cmap=plt.cm.binary)
    plt.xlabel(class_names[train_labels[i]])
plt.show()

---

## 5. Building the Model (MLP)

We will use a standard feed-forward Dense Neural Network (MLP). 

### 5.1 Flattening
Since Dense layers expect 1D vectors as input, we must first **flatten** our 2D (28x28) images into a 1D vector of 784 pixels ($28 \times 28 = 784$).

![Flattening Diagram](https://miro.medium.com/max/1400/1*GLQjM9k0gZ14qWW8dqglsA.png)
*Figure: Flattening a 2D matrix into a 1D vector.*
[Image Source](https://medium.com/)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

model = Sequential([
    # Layer 1: Flatten the 28x28 images to a 1D vector of 784 pixels
    Flatten(input_shape=(28, 28)),
    
    # Layer 2: Dense hidden layer with 128 neurons
    Dense(128, activation='relu'),
    
    # Layer 3: Dense output layer with 10 neurons (one for each class)
    # Softmax turns the output into a probability distribution
    Dense(10, activation='softmax')
])

model.summary()

### 5.2 Compiling and Training
We use `adam` optimizer and `sparse_categorical_crossentropy` loss (because our labels are integers, not yet one-hot encoded; Keras handles this automatically with 'sparse').

In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
history = model.fit(train_images, train_labels, epochs=5, validation_split=0.1)

In [ ]:
# Plot accuracy
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.5, 1])
plt.legend(loc='lower right')
plt.show()

---
## 6. The Limitation of MLPs for Images

While our MLP achieved decent accuracy, there is a fundamental flaw in using Dense networks for image processing:

1.  **Loss of Spatial Structure:** The `Flatten` layer destroys the 2D grid structure. A pixel at (0,0) is far away from (0,1) in the flattened vector. The network loses the concept of "up", "down", "left", "right".
2.  **No Shift Invariance:** If we shift the cat in the image slightly to the left, the flattened vector changes completely. The MLP has to relearn the cat in the new position from scratch.
3.  **Parameter Explosion:** For larger images (e.g., 200x200 color image), a single Dense layer would require billions of weights, making it impossible to train.

### The Solution: Convolutional Neural Networks (CNNs)

In the next chapter, we will introduce **CNNs**. Instead of flattening the image, CNNs use specialized **filters** (kernels) that slide across the 2D image to detect features like edges, textures, and shapes, preserving the spatial relationships.

**Preview:**
A CNN behaves like a flashlight scanning over the image, looking for specific patterns regardless of where they are.

In [ ]:
# Final Exercise to test understanding
print("Exercise: Why would this MLP fail on a high-resolution photo of a landscape?")
# Answer: Due to the massive number of parameters required and the loss of local spatial patterns.